# State and input discretization by policy iteration

A port of `statediscPI.mlx`.

The minimum-energy problem of Unit 7 again: drive a double integrator from
rest at the origin to position $1$ with zero velocity, minimising
$\int_0^1u^2\,dt$. Unit 7 discretized state, input and time and solved the
resulting grid problem by backward dynamic programming; here the same grid
problem is solved by **policy iteration**, and the two are compared directly.

The point is that they agree. Policy iteration and dynamic programming are
different routes to the same optimal policy, and on a problem this size the
agreement can be checked exactly rather than argued.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (14, 4)

from policy_iteration import policy_iteration, dp

In [ ]:
dt = 0.5
K  = round(1/dt)
du = 0.2
dv = dt*du
dd = dt**2/2*du

n_u = int(2*(7/du) + 1)
n_v = round(2/dv)
n_d = round(1/dd)
n_x = (n_d+1)*(n_v+1)
print(f"{n_x} states, {n_u} inputs, {K} stages")

In [ ]:
idx    = np.arange(n_x).reshape(-1, 1)
ones_u = np.ones((1, n_u))
u_grid = np.arange(-(n_u-1)//2, (n_u-1)//2 + 1).reshape(1, -1)

D   = (idx // (n_v+1)) @ ones_u
V   = (idx %  (n_v+1)) @ ones_u
U   = np.ones((n_x, 1)) @ u_grid
V_1 = np.clip(V + U, 0, n_v)
D_1 = np.clip(D + 2*V + U, 0, n_d)

M  = (V_1 + (n_v+1)*D_1).astype(int)     # next state
C  = np.ones((n_x, 1)) @ (u_grid**2)     # stage cost
Jh = np.full(n_x, np.inf)
Jh[n_d*(n_v+1)] = 1.0                    # only the target state is admissible

In [ ]:
# the same model, in the generic MDP form the policy-iteration driver expects
DM = [[[[int(M[i, j])]    for j in range(n_u)] for i in range(n_x)] for _ in range(K)]
CM = [[[[float(C[i, j])]  for j in range(n_u)] for i in range(n_x)] for _ in range(K)]
PM = [[[[1.0]             for j in range(n_u)] for i in range(n_x)] for _ in range(K)]
mdp = {"DM": DM, "CM": CM, "PM": PM, "gh": list(Jh)}

mu, J, history = policy_iteration(mdp)
u_dp, J_dp = dp(M, C, Jh, K)

print(f"policy iteration: {len(history)} iterations, J[0] = {J[0][0]}")
print(f"dynamic programming:              J[0] = {J_dp[0, 0]}")
print(f"they agree: {np.isclose(J[0][0], J_dp[0, 0])}")
print(f"\ncost = dt * J * du^2 = {dt*J[0][0]*du**2:.4f}")

Both give $801$, hence a cost of $16.02$.

Unit 7's grid DP gives $16.00$, and the difference is worth a word. The
terminal cost here is `Jh[target] = 1` with $+\infty$ everywhere else: the
$1$ is a marker saying "this state is admissible", not a real cost. It
contributes $dt\cdot1\cdot du^2 = 0.5\times1\times0.04 = 0.02$. Setting it to
$0$ instead reproduces $16.00$ exactly.

In [ ]:
d = np.zeros(K+1); v = np.zeros(K+1); u = np.zeros(K)
xbar = np.zeros(K+1, dtype=int)
for k in range(K):
    ubar = mu[k][xbar[k]]
    xbar[k+1] = M[xbar[k], ubar]
    u[k] = (ubar - (n_u-1)/2)*du
    d[k+1] = (xbar[k+1] // (n_v+1))*dd
    v[k+1] = (xbar[k+1] %  (n_v+1))*dv
print("u =", u, "  position =", d, "  velocity =", v)

In [ ]:
t = np.linspace(0, 1, 1001)
z_sol, v_sol, u_sol = 3*t**2 - 2*t**3, 6*t - 6*t**2, 6 - 12*t
kvec = np.arange(K+1)/K

fig, ax = plt.subplots(1, 3)
ax[0].stem(kvec[:-1], u); ax[0].plot(t, u_sol, "g"); ax[0].legend(["$u_k$", "$u(t)$"])
ax[1].stem(kvec, d);      ax[1].plot(t, z_sol, "g"); ax[1].legend(["$x_k$", "$x(t)$"])
ax[2].stem(kvec, v);      ax[2].plot(t, v_sol, "g"); ax[2].legend(["$v_k$", "$v(t)$"])
for a in ax: a.set_xlabel("time"); a.grid(True)
plt.tight_layout(); plt.show()

The policy gives $u=[4,-4]$ and positions $[0,\,0.5,\,1]$ -- the same
trajectory Unit 7's DP found, and the same one the Q-learning notebook in
`code/unit10/` recovered without a model. Three methods, one answer.